In [1]:
# Read raw_job_posting.txt
with open("raw_job_posting.txt", "r") as f:
    raw_job_posting = f.read()

In [2]:
import os

from langgraph.checkpoint.postgres.aio import AsyncPostgresSaver
from langgraph.graph import END, START, StateGraph
from psycopg.rows import dict_row
from psycopg_pool import AsyncConnectionPool

from scorecard.nodes.node_generate_questions import node_generate_questions
from scorecard.state import ScorecardGraphState, ScorecardInputGraphState
from scorecard.sub_graph.enrichment.graph import get_enrichment_graph

connection_kwargs = {
    "autocommit": True,
    "prepare_threshold": 0,
    "row_factory": dict_row,
}

async with AsyncConnectionPool(
    # Example configuration
    conninfo=f"{os.environ['DATABASE_URL']}",
    max_size=20,
    kwargs=connection_kwargs,
) as pool:
    checkpointer = AsyncPostgresSaver(pool)
    await checkpointer.setup()

    # Define a new graph
    workflow = StateGraph(ScorecardGraphState, input=ScorecardInputGraphState)
    workflow.add_node("agent_enrichment", get_enrichment_graph())
    workflow.add_node("generate_questions", node_generate_questions)
    workflow.add_edge(START, "agent_enrichment")
    workflow.add_edge("agent_enrichment", "generate_questions")
    workflow.add_edge("generate_questions", END)
    graph = workflow.compile(checkpointer=checkpointer)


    config = {"configurable": {"thread_id": "6"}}
    res = await graph.ainvoke({"raw_job_posting": raw_job_posting}, config=config)
    print(res)

/Users/aberman/Documents/Workshop/repio-intelligence/.venv/lib/python3.12/site-packages/langsmith/client.py:5301: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  prompt = loads(json.dumps(prompt_object.manifest))
/Users/aberman/Documents/Workshop/repio-intelligence/src/scorecard/nodes/node_generate_questions.py:13: LangChainBetaWarning: The function `init_chat_model` is in beta. It is actively being worked on, so the API may change.
  model = init_chat_model(


{'raw_job_posting': "Ynstant\nOperations Manager CDI Paris Salaire :52K à 72K\xa0€ Début :19 août 2024 Télétravail occasionnel Expérience :> 2 ans Compétences & expertises Compétences en communication Outils d'automatisation Sensibilité culturelle Pandas Sql\nentreprise : Ynstant\n\nYnstant est une startup qui révolutionne la mobilité du quotidien pour la rendre plus durable, grâce à une application de covoiturage instantané. Concrètement, Ynstant permet aux conducteurs de covoiturer en deux clics, sans détour et au dernier moment avant de partir, préservant ainsi leur flexibilité.\n\nDescriptif du poste\nMission\xa0: Le Ops Manager sera responsable de la gestion des opérations quotidiennes des Certificats d’Economie d’Energie (CEE) en s’appuyant sur de la data. Les missions sont :\n-Monitoring des dossiers CEE et optimisation de chaque étape du parcours\n-Traitement des dossiers présentant des anomalies\n-Conformité des dossiers CEE à la réglementation\nProfil recherché\nCOMPÉTENCES T

In [4]:
from pprint import pprint

# new line at every question
for question in res["questions"]:
    print("\n")
    pprint(question, indent=2)



( 'questions',
  [ Question(question='Could you elaborate on the specific types of automation tools or scripts that would be beneficial for this role?', criteria_type=<CriteriaType.HARD_SKILL: 'HARD_SKILL'>, answer=None),
    Question(question='What level of proficiency in SQL and Python is expected for this position?', criteria_type=<CriteriaType.HARD_SKILL: 'HARD_SKILL'>, answer=None),
    Question(question="Are there particular cultural sensitivities or aspects that are especially important for this role, given the company's focus on sustainability and mobility?", criteria_type=<CriteriaType.SOFT_SKILL: 'SOFT_SKILL'>, answer=None),
    Question(question='What specific challenges or pressures should the candidate be prepared to handle in this role?', criteria_type=<CriteriaType.SOFT_SKILL: 'SOFT_SKILL'>, answer=None),
    Question(question="How does the company define 'intellectual honesty' and 'humility' in the context of its work culture?", criteria_type=<CriteriaType.SOFT_SKILL: